# UC-PROC-1 — Execute a Process Synchronously

**Persona:** Scientist

**Goal:** List available OGC API Processes, inspect a process schema, execute it
synchronously using `Prefer: wait=30`, and assert the result arrives in the same response.

**Prerequisites:**
- OGC API Processes extension registered and reachable
- Process `geometry-validator` (or the value of `PROCESS_ID`) registered and able to
  complete within 30 seconds
- Platform built with synchronous execution enabled (`Prefer: wait=N` supported)

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`  
**Optional:** `PROCESS_ID` (defaults to `geometry-validator`)

In [ ]:
import os
import json
import time

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ["DYNASTORE_BASE_URL"]
TOKEN = os.environ["DYNASTORE_TOKEN"]
PROCESS_ID = os.environ.get("PROCESS_ID", "geometry-validator")

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

# job_id populated later if async fallback is needed
job_id = None

print(f"Connected to {BASE_URL}")
print(f"PROCESS_ID={PROCESS_ID}")

In [ ]:
# List all available processes
r = client.get("/processes/processes")
print(r.status_code)
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

body = r.json()
processes = body.get("processes", body) if isinstance(body, dict) else body
if isinstance(processes, dict):
    processes = list(processes.values())

print(f"Available processes ({len(processes)}):")
for proc in processes:
    proc_id = proc.get("id", proc.get("name", "?"))
    title = proc.get("title", proc.get("summary", ""))
    print(f"  {proc_id}: {title}")

In [ ]:
# Inspect the process schema
r = client.get(f"/processes/processes/{PROCESS_ID}")
print(r.status_code)
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

proc_schema = r.json()
print(f"Process: {proc_schema.get('id')} — {proc_schema.get('title', '')}")
print("\nInputs:")
for name, defn in proc_schema.get("inputs", {}).items():
    schema = defn.get("schema", {})
    print(f"  {name}: type={schema.get('type', '?')} required={defn.get('minOccurs', 0) > 0}")

print("\nOutputs:")
for name, defn in proc_schema.get("outputs", {}).items():
    schema = defn.get("schema", {})
    print(f"  {name}: type={schema.get('type', '?')}")

In [ ]:
# Execute synchronously: Prefer: wait=30 requests the server to block up to 30 s
# and return the result directly (200 or 201) rather than a job receipt (202).
sync_payload = {
    "inputs": {
        "geometry": {
            "type": "Polygon",
            "coordinates": [[[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]]],
        }
    },
    "outputs": {"report": {"transmissionMode": "value"}},
}

r = client.post(
    f"/processes/processes/{PROCESS_ID}/execution",
    content=json.dumps(sync_payload),
    headers={"Prefer": "wait=30"},
)
print(r.status_code)
assert r.status_code in (200, 201, 202), (
    f"Expected 200/201 (sync result) or 202 (async fallback), got {r.status_code}: {r.text}"
)

result = r.json()
status = result.get("status", "")
print(f"status={status}")

if r.status_code in (200, 201):
    assert status == "successful", f"Expected status='successful', got '{status}': {result}"
    print("Synchronous execution successful.")
else:
    # Server fell back to async; save job_id for poll cell below
    job_id = result.get("jobID", result.get("job_id", ""))
    assert job_id, f"202 returned but no jobID in response: {result}"
    print(f"Async fallback: job_id={job_id} (will poll below)")

In [ ]:
# Print result if available in the synchronous response
if r.status_code in (200, 201):
    assert "outputs" in result or "results" in result, (
        f"Expected 'outputs' or 'results' key in sync result, got keys: {list(result.keys())}"
    )
    output_data = result.get("outputs") or result.get("results", {})
    print("Result outputs:")
    print(json.dumps(output_data, indent=2)[:800])
else:
    print(f"Result not yet available (async job {job_id}). See poll cell below.")

## Edge Cases

In [ ]:
# Edge case 1: process exceeds wait= timeout -> 202 + poll loop
# If the server cannot complete within the requested wait window it returns 202 and
# a job receipt. This cell handles that path whether triggered by a slow process
# or by the async-fallback branch above.

if job_id:
    print(f"Polling job {job_id} (async fallback path) ...")
    status = "accepted"
    for attempt in range(20):
        resp = client.get(f"/processes/jobs/{job_id}")
        assert resp.status_code == 200, (
            f"Expected 200 polling job, got {resp.status_code}: {resp.text}"
        )
        status = resp.json().get("status", "")
        print(f"  Attempt {attempt + 1}: status={status}")
        if status in ("successful", "failed", "dismissed"):
            break
        time.sleep(min(2 ** attempt, 30))

    assert status == "successful", (
        f"Job did not reach 'successful'; final status='{status}'"
    )
    print("Async job completed successfully via poll.")
else:
    print("Synchronous path used; no polling required.")

In [ ]:
# Edge case 2: Prefer: respond-async forces 202 regardless of process duration
r_async = client.post(
    f"/processes/processes/{PROCESS_ID}/execution",
    content=json.dumps(sync_payload),
    headers={"Prefer": "respond-async"},
)
print(r_async.status_code)
assert r_async.status_code == 202, (
    f"Expected 202 with Prefer: respond-async, got {r_async.status_code}: {r_async.text}"
)

forced_async_result = r_async.json()
forced_status = forced_async_result.get("status", "")
assert forced_status == "accepted", (
    f"Expected status='accepted' for forced async, got '{forced_status}'"
)
forced_job_id = forced_async_result.get("jobID", forced_async_result.get("job_id", ""))
assert forced_job_id, f"No jobID in forced-async response: {forced_async_result}"
print(f"Prefer: respond-async -> 202 accepted, job_id={forced_job_id}")

In [ ]:
# Edge case 3: no Prefer header -> server default
# OGC API Processes Part 1 leaves the default behaviour implementation-defined.
# DynaStore defaults to synchronous (200/201) for fast processes and async (202)
# for processes that declare an estimated duration above the platform threshold.
r_default = client.post(
    f"/processes/processes/{PROCESS_ID}/execution",
    content=json.dumps(sync_payload),
)
print(r_default.status_code)
assert r_default.status_code in (200, 201, 202), (
    f"Expected 200, 201, or 202 with no Prefer header, got {r_default.status_code}: {r_default.text}"
)
default_body = r_default.json()
print(f"No Prefer header: status_code={r_default.status_code}  body_status={default_body.get('status', '?')}")
default_job_id = default_body.get("jobID", default_body.get("job_id"))

## Teardown

In [ ]:
# Dismiss any async jobs created during this notebook
jobs_to_dismiss = [
    jid for jid in [job_id, forced_job_id, default_job_id] if jid
]

for jid in jobs_to_dismiss:
    r_del = client.delete(f"/processes/jobs/{jid}")
    print(r_del.status_code, jid)
    assert r_del.status_code in (200, 204), (
        f"Expected 200 or 204 dismissing job {jid}, got {r_del.status_code}: {r_del.text}"
    )
    print(f"  Job {jid} dismissed.")

if not jobs_to_dismiss:
    print("No async jobs to dismiss (synchronous path used throughout).")

client.close()